4.0.1 Drive i putanje

Montira se Drive i postavljaju se putanje ka curated podacima i folderu za rezultate. Ovaj notebook kreira Runs/qc_YYYYMMDD_HHMMSS i u njega upisuje izveštaje i slike.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, math, random, hashlib
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Diplomski")
CURATED = ROOT / "Data" / "curated"
RUNS = ROOT / "Runs"
RUNS.mkdir(parents=True, exist_ok=True)

run_id = time.strftime("%Y%m%d_%H%M%S")
OUT = RUNS / f"qc_{run_id}"
OUT.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("CURATED:", CURATED)
print("OUT:", OUT)

Mounted at /content/drive
ROOT: /content/drive/MyDrive/Diplomski
CURATED: /content/drive/MyDrive/Diplomski/Data/curated
OUT: /content/drive/MyDrive/Diplomski/Runs/qc_20260313_095355


4.0.2 Učitavanje splitova i meta.json

Učitavaju se train/val/test CSV i meta.json, a kolone za label i putanju se automatski pronalaze. Ovo obezbeđuje isti QC kod za LC25000, SipakMed i RM1000.

In [3]:
import pandas as pd
import numpy as np

def load_meta(ds):
    with open(CURATED / ds / "meta.json", "r", encoding="utf-8") as f:
        return json.load(f)

def load_split(ds, split):
    return pd.read_csv(CURATED / ds / f"{split}.csv")

def infer_task(meta):
    t = meta.get("task_type")
    if t in ["image", "tabular"]:
        return t
    if "image" in json.dumps(meta).lower():
        return "image"
    return "tabular"

def label_col(meta, df):
    c = meta.get("label_col")
    if c and c in df.columns:
        return c
    for cand in ["label", "target", "y", "class", "Recurred"]:
        if cand in df.columns:
            return cand
    return df.columns[-1]

def path_col(meta, df):
    c = meta.get("path_col")
    if c and c in df.columns:
        return c
    for cand in ["path", "filepath", "image_path", "img_path", "file"]:
        if cand in df.columns:
            return cand
    return None

IMAGE_DATASETS = ["lc25000", "sipakmed", "rm1000_lung_history"]

In [4]:
import torch
print("cuda:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cuda: True
device: NVIDIA L4


4.1 Exact duplicate provera (hash) unutar svakog split-a

Ovde tražimo identične fajlove po MD5 hashu da bismo detektovali duplikate. Rezultat se snima kao hash_duplicates.csv po datasetu.

In [5]:
import hashlib
from tqdm.auto import tqdm

def md5_file(path, chunk=1024*1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def sample_paths(df, pcol, max_files=None, seed=42):
    paths = df[pcol].astype(str).tolist()
    if (max_files is None) or (len(paths) <= max_files):
        return paths
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(paths), size=max_files, replace=False)
    return [paths[i] for i in idx]

def hash_paths(paths, desc):
    m = {}
    missing = 0
    for p in tqdm(paths, desc=desc):
        try:
            h = md5_file(p)
            m.setdefault(h, []).append(p)
        except:
            missing += 1
    return m, missing

def md5_train_test_leakage(ds, max_train=None, max_test=None, seed=42):
    meta = load_meta(ds)
    df_tr = load_split(ds, "train")
    pcol = path_col(meta, df_tr)

    tr_paths = sample_paths(df_tr, pcol, max_train, seed)
    te_paths = sample_paths(load_split(ds, "test"), pcol, max_test, seed+1)

    tr_map, tr_missing = hash_paths(tr_paths, f"{ds} md5 train")
    te_map, te_missing = hash_paths(te_paths, f"{ds} md5 test")

    common = set(tr_map.keys()) & set(te_map.keys())
    leak_pairs = []
    for h in list(common)[:5000]:
        for p in tr_map[h]:
            for q in te_map[h]:
                leak_pairs.append({"dataset": ds, "hash": h, "train_path": p, "test_path": q})

    return {
        "dataset": ds,
        "pcol": pcol,
        "train_hashed": len(tr_paths),
        "test_hashed": len(te_paths),
        "train_missing": tr_missing,
        "test_missing": te_missing,
        "common_hashes": len(common),
        "leak_pairs": leak_pairs
    }

md5_plan = {
    "lc25000": {"max_train": 5000, "max_test": 2000},
    "rm1000_lung_history": {"max_train": 5000, "max_test": 2000},
    "sipakmed": {"max_train": None, "max_test": None}
}

md5_results = []
all_leaks = []
for ds, cfg in md5_plan.items():
    res = md5_train_test_leakage(ds, **cfg, seed=42)
    md5_results.append({k: res[k] for k in ["dataset","train_hashed","test_hashed","train_missing","test_missing","common_hashes"]})
    all_leaks += res["leak_pairs"]
    print(res["dataset"], "common_hashes:", res["common_hashes"])

df_md5_summary = pd.DataFrame(md5_results)
df_md5_summary.to_csv(OUT / "md5_train_test_summary.csv", index=False)
pd.DataFrame(all_leaks).to_csv(OUT / "md5_train_test_leaks.csv", index=False)

print("Saved:", OUT / "md5_train_test_summary.csv")
print("Saved:", OUT / "md5_train_test_leaks.csv")
df_md5_summary

lc25000 md5 train:   0%|          | 0/5000 [00:00<?, ?it/s]

lc25000 md5 test:   0%|          | 0/2000 [00:00<?, ?it/s]

lc25000 common_hashes: 41


rm1000_lung_history md5 train:   0%|          | 0/5000 [00:00<?, ?it/s]

rm1000_lung_history md5 test:   0%|          | 0/2000 [00:00<?, ?it/s]

rm1000_lung_history common_hashes: 84


sipakmed md5 train:   0%|          | 0/2835 [00:00<?, ?it/s]

sipakmed md5 test:   0%|          | 0/606 [00:00<?, ?it/s]

sipakmed common_hashes: 0
Saved: /content/drive/MyDrive/Diplomski/Runs/qc_20260313_095355/md5_train_test_summary.csv
Saved: /content/drive/MyDrive/Diplomski/Runs/qc_20260313_095355/md5_train_test_leaks.csv


,dataset,train_hashed,test_hashed,train_missing,test_missing,common_hashes
0,lc25000,5000,2000,0,0,41
1,rm1000_lung_history,5000,2000,0,0,84
2,sipakmed,2835,606,0,0,0


4.2 Leakage provera: isti hash preko splitova

Ovde proveravamo da li se identična slika pojavljuje u više splitova (npr. train i test). Ako ima konflikata, to je klasično curenje podataka.

In [6]:
def hash_map(df, pcol):
    m = {}
    for p in df[pcol].astype(str).tolist():
        try:
            h = md5_file(p)
            m.setdefault(h, []).append(p)
        except:
            pass
    return m

leak_rows = []
for ds in IMAGE_DATASETS:
    meta = load_meta(ds)
    df_tr = load_split(ds, "train")
    pcol = path_col(meta, df_tr)

    maps = {}
    for sp in ["train", "val", "test"]:
        maps[sp] = hash_map(load_split(ds, sp), pcol)

    common = set(maps["train"].keys()) & (set(maps["val"].keys()) | set(maps["test"].keys()))
    for h in sorted(list(common)):
        for p in maps["train"].get(h, []):
            leak_rows.append({"dataset": ds, "hash": h, "split_a": "train", "path_a": p})
        for sp in ["val","test"]:
            for p in maps[sp].get(h, []):
                leak_rows.append({"dataset": ds, "hash": h, "split_a": sp, "path_a": p})

df_leak = pd.DataFrame(leak_rows)
df_leak.to_csv(OUT / "leakage_exact_hash.csv", index=False)
print("Saved:", OUT / "leakage_exact_hash.csv", "rows:", len(df_leak))
df_leak.head(10)

Saved: /content/drive/MyDrive/Diplomski/Runs/qc_20260313_095355/leakage_exact_hash.csv rows: 1744


,dataset,hash,split_a,path_a
0,lc25000,00d43876efc9e7d0a61f8debc72bd12f,train,/content/drive/MyDrive/Diplomski/Data/raw/lc25...
1,lc25000,00d43876efc9e7d0a61f8debc72bd12f,test,/content/drive/MyDrive/Diplomski/Data/raw/lc25...
2,lc25000,00e2ed61a39b58288b62665086cf0590,train,/content/drive/MyDrive/Diplomski/Data/raw/lc25...
3,lc25000,00e2ed61a39b58288b62665086cf0590,val,/content/drive/MyDrive/Diplomski/Data/raw/lc25...
4,lc25000,013ef0d1c5f73bb61210fbac0dd30929,train,/content/drive/MyDrive/Diplomski/Data/raw/lc25...
5,lc25000,013ef0d1c5f73bb61210fbac0dd30929,test,/content/drive/MyDrive/Diplomski/Data/raw/lc25...
6,lc25000,018d46a79a066ef9d3ca551c96168ea5,train,/content/drive/MyDrive/Diplomski/Data/raw/lc25...
7,lc25000,018d46a79a066ef9d3ca551c96168ea5,test,/content/drive/MyDrive/Diplomski/Data/raw/lc25...
8,lc25000,02aab8fda950cd348933a91bac1e1b04,train,/content/drive/MyDrive/Diplomski/Data/raw/lc25...
9,lc25000,02aab8fda950cd348933a91bac1e1b04,val,/content/drive/MyDrive/Diplomski/Data/raw/lc25...


4.3 QC metrike: blur i brightness

Ovde računamo jednostavne QC signale po slici koji pomažu da se detektuju loši uzorci. Snimamo qc_scores.csv i primer-grid slika.

In [7]:
import cv2
import matplotlib.pyplot as plt
from PIL import Image

def qc_scores(path):
    try:
        im = cv2.imread(path)
        if im is None:
            return None
        gray = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
        blur = cv2.Laplacian(gray, cv2.CV_64F).var()
        bright = float(gray.mean())
        return blur, bright
    except:
        return None

qc_rows = []
for ds in IMAGE_DATASETS:
    meta = load_meta(ds)
    df_tr = load_split(ds, "train")
    pcol = path_col(meta, df_tr)

    for sp in ["train", "val", "test"]:
        df = load_split(ds, sp)
        for p in df[pcol].astype(str).tolist():
            s = qc_scores(p)
            if s is None:
                qc_rows.append({"dataset": ds, "split": sp, "path": p, "blur": np.nan, "brightness": np.nan})
            else:
                blur, bright = s
                qc_rows.append({"dataset": ds, "split": sp, "path": p, "blur": blur, "brightness": bright})

df_qc = pd.DataFrame(qc_rows)
df_qc.to_csv(OUT / "qc_scores.csv", index=False)
print("Saved:", OUT / "qc_scores.csv", "rows:", len(df_qc))
df_qc.head(5)

Saved: /content/drive/MyDrive/Diplomski/Runs/qc_20260313_095355/qc_scores.csv rows: 44049


,dataset,split,path,blur,brightness
0,lc25000,train,/content/drive/MyDrive/Diplomski/Data/raw/lc25...,27.173692,110.245309
1,lc25000,train,/content/drive/MyDrive/Diplomski/Data/raw/lc25...,18.401650,154.235192
2,lc25000,train,/content/drive/MyDrive/Diplomski/Data/raw/lc25...,59.640405,199.082735
3,lc25000,train,/content/drive/MyDrive/Diplomski/Data/raw/lc25...,25.640413,162.490711
4,lc25000,train,/content/drive/MyDrive/Diplomski/Data/raw/lc25...,9.277871,182.041161


4.4 Flagovanje outlier-a i pravljenje curated_qc

Ovde definišemo pragove po datasetu i izbacujemo očigledno loše uzorke (ili ih samo označimo). Export se snima u Data/curated_qc.

In [8]:
CURATED_QC = ROOT / "Data" / "curated_qc"
CURATED_QC.mkdir(parents=True, exist_ok=True)

def apply_qc_and_export(ds, blur_q=0.02, bright_low_q=0.01, bright_high_q=0.99, drop=True):
    meta = load_meta(ds)
    base = CURATED / ds
    outd = CURATED_QC / ds
    outd.mkdir(parents=True, exist_ok=True)
    (outd / "meta.json").write_text((base / "meta.json").read_text(encoding="utf-8"), encoding="utf-8")

    df_tr = load_split(ds, "train")
    pcol = path_col(meta, df_tr)

    df_ds = df_qc[df_qc["dataset"] == ds].dropna()
    blur_thr = float(df_ds["blur"].quantile(blur_q))
    b_lo = float(df_ds["brightness"].quantile(bright_low_q))
    b_hi = float(df_ds["brightness"].quantile(bright_high_q))

    summary = []
    for sp in ["train","val","test"]:
        df = load_split(ds, sp)
        qc_sp = df_qc[(df_qc["dataset"] == ds) & (df_qc["split"] == sp)][["path","blur","brightness"]]
        merged = df.merge(qc_sp, left_on=pcol, right_on="path", how="left")
        merged["flag_blur"] = merged["blur"] < blur_thr
        merged["flag_dark"] = merged["brightness"] < b_lo
        merged["flag_bright"] = merged["brightness"] > b_hi
        merged["flag_any"] = merged[["flag_blur","flag_dark","flag_bright"]].any(axis=1)

        before = len(merged)
        if drop:
            merged_out = merged[~merged["flag_any"]].copy()
        else:
            merged_out = merged.copy()

        after = len(merged_out)
        merged_out.drop(columns=["path","blur","brightness"], errors="ignore").to_csv(outd / f"{sp}.csv", index=False)

        summary.append({"dataset": ds, "split": sp, "before": before, "after": after, "dropped": before-after,
                        "blur_thr": blur_thr, "bright_lo": b_lo, "bright_hi": b_hi})

    return pd.DataFrame(summary)

qc_summaries = []
for ds in IMAGE_DATASETS:
    qc_summaries.append(apply_qc_and_export(ds, drop=True))

df_qc_sum = pd.concat(qc_summaries, ignore_index=True)
df_qc_sum.to_csv(OUT / "qc_export_summary.csv", index=False)
print("Saved:", OUT / "qc_export_summary.csv")
df_qc_sum

Saved: /content/drive/MyDrive/Diplomski/Runs/qc_20260313_095355/qc_export_summary.csv


,dataset,split,before,after,dropped,blur_thr,bright_lo,bright_hi
0,lc25000,train,17500,16804,696,8.256629,115.941586,211.092240
1,lc25000,val,3750,3605,145,8.256629,115.941586,211.092240
2,lc25000,test,3750,3591,159,8.256629,115.941586,211.092240
3,sipakmed,train,2835,2750,85,10.835325,61.857525,214.373514
4,sipakmed,val,608,593,15,10.835325,61.857525,214.373514
5,sipakmed,test,606,577,29,10.835325,61.857525,214.373514
6,rm1000_lung_history,train,10500,10130,370,7.669642,111.706022,189.479042
7,rm1000_lung_history,val,2250,2158,92,7.669642,111.706022,189.479042
8,rm1000_lung_history,test,2250,2157,93,7.669642,111.706022,189.479042


4.5 Mini “QC vs no-QC” trening (samo SipakMed)

Ovde proveravamo da li QC pomaže ili odmaže, bez trošenja resursa na sve skupove. Snimamo rezultate u qc_vs_noqc.json.

In [9]:
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

class ImageCsvDatasetSimple(torch.utils.data.Dataset):
    def __init__(self, df, pcol, ycol, label_to_idx=None, tfm=None):
        self.df = df.reset_index(drop=True)
        self.pcol = pcol
        self.ycol = ycol
        self.tfm = tfm
        labels = self.df[ycol].astype(str).tolist()
        if label_to_idx is None:
            uniq = sorted(list(set(labels)))
            self.label_to_idx = {u:i for i,u in enumerate(uniq)}
        else:
            self.label_to_idx = label_to_idx
        self.y = [self.label_to_idx[str(x)] for x in labels]

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        p = str(self.df.iloc[i][self.pcol])
        y = self.y[i]
        im = Image.open(p).convert("RGB")
        if self.tfm: im = self.tfm(im)
        return im, y

def quick_train_eval(curated_root, ds="sipakmed", seed=42, epochs=1, img_size=224, batch_size=32):
    set_seed(seed)
    meta = json.loads((curated_root / ds / "meta.json").read_text(encoding="utf-8"))
    df_tr = pd.read_csv(curated_root / ds / "train.csv")
    df_va = pd.read_csv(curated_root / ds / "val.csv")
    df_te = pd.read_csv(curated_root / ds / "test.csv")

    ycol = label_col(meta, df_tr)
    pcol = path_col(meta, df_tr)

    tfm_tr = transforms.Compose([transforms.Resize((img_size,img_size)), transforms.RandomHorizontalFlip(0.5), transforms.ToTensor()])
    tfm_ev = transforms.Compose([transforms.Resize((img_size,img_size)), transforms.ToTensor()])

    ds_tr = ImageCsvDatasetSimple(df_tr, pcol, ycol, None, tfm_tr)
    ds_va = ImageCsvDatasetSimple(df_va, pcol, ycol, ds_tr.label_to_idx, tfm_ev)
    ds_te = ImageCsvDatasetSimple(df_te, pcol, ycol, ds_tr.label_to_idx, tfm_ev)

    dl_tr = torch.utils.data.DataLoader(ds_tr, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available())
    dl_va = torch.utils.data.DataLoader(ds_va, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())
    dl_te = torch.utils.data.DataLoader(ds_te, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())

    m = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
    m.fc = nn.Linear(m.fc.in_features, len(ds_tr.label_to_idx))
    m = m.to(device)

    opt = torch.optim.AdamW(m.parameters(), lr=3e-4, weight_decay=1e-2)
    loss_fn = nn.CrossEntropyLoss()

    for _ in range(epochs):
        m.train()
        for x, y in dl_tr:
            x = x.to(device)
            y = y.to(device)
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(m(x), y)
            loss.backward()
            opt.step()

    def eval_dl(dl):
        m.eval()
        ys, ps = [], []
        with torch.no_grad():
            for x, y in dl:
                x = x.to(device)
                y = y.to(device)
                pred = torch.argmax(m(x), dim=1)
                ys += y.cpu().numpy().tolist()
                ps += pred.cpu().numpy().tolist()
        return float(accuracy_score(ys, ps)), float(f1_score(ys, ps, average="macro"))

    return {"val": eval_dl(dl_va), "test": eval_dl(dl_te), "n_train": len(ds_tr), "n_test": len(ds_te)}